# SchemaShift GRPO Training - Kaggle T4

**Stage 1 / Stage 2 team split (TOS-compliant parallel experiments):**

- **Account 1 (Yashash - STAGE 1 MAIN):** Shaped reward (rubric + dense + gates). **Running tonight.** Pipeline-validation run - go/no-go for Stage 2.
- **Account 2 (Gajanand - Stage 2 parallel):** Shaped reward (same config as Account 1). Runs after Stage 1 convergence - a second seed on the same recipe for variance check.
- **Account 3 (Likith - curriculum ablation):** Shaped reward + procedural drift scheduler (see `build_procedural_scenarios`). Alternative: swap base model to `Qwen/Qwen2.5-Coder-1.5B-Instruct` for a coder-model comparison.

**Before running (all accounts):**
- Set Kaggle Secret: `HF_TOKEN` (write-scoped; pushes checkpoints)
- Set Kaggle Secret: `SCHEMASHIFT_URL` = `https://yashash045-schemashift.hf.space`
- Set Kaggle Secret: `HF_USERNAME` (your HF namespace; falls back to `yashash045` if unset)
- Accelerator: GPU T4 x2 (P100 fallback). Internet: ON. Persistence: ON.

**Stage 1 runbook:** see `training/STAGE_1_KAGGLE_RUNBOOK.md` for the full step-by-step guide.

**Dependency baseline:** TRL 0.29.0 + Unsloth 2026.x + Kaggle-native torch 2.10.
No vLLM (incompatible with Kaggle's torch 2.10). Reference implementation: [sid-rp/kube-sre-gym](https://github.com/sid-rp/kube-sre-gym) (previous hackathon winner, TRL 0.29 on H100 with vLLM). We use their TRL version but keep rollouts inside `reward_fn` since `trl.experimental.openenv.generate_rollout_completions` requires `use_vllm=True`.


In [ ]:
# Install deps - TRL 0.29.0 without vLLM (Kaggle torch 2.10 native).
# Mirrors winner repo sid-rp/kube-sre-gym (trl==0.29.0, peft) minus vLLM.
# TRL 0.29.0 is clean on imports (no mergekit/vllm eager-import crash).
!pip install -q unsloth
!pip install -q "trl==0.29.0" peft
!pip install -q httpx pydantic fastapi openai

# Verify critical deps. If any print wrong versions, STOP.
import torch
print(f"torch: {torch.__version__}")
print(f"cuda: {torch.cuda.is_available()}")

import unsloth, trl, peft
print(f"unsloth: {unsloth.__version__}")
print(f"trl:     {trl.__version__}")
print(f"peft:    {peft.__version__}")
print("Core deps verified (TRL 0.29.0, no vLLM).")


In [ ]:
# Clone repo + install editable
import os
os.chdir("/kaggle/working")
!git clone https://github.com/Yashash4/SchemaShift.git || (cd SchemaShift && git pull)
os.chdir("/kaggle/working/SchemaShift")
!pip install -e .

In [ ]:
# Env health check - fail fast if the HF Space is unreachable.
# Reads SCHEMASHIFT_URL from Kaggle secrets. If this cell raises, DO NOT proceed to training.
import os
from client import SchemaShiftEnvClient

SCHEMASHIFT_URL = os.getenv("SCHEMASHIFT_URL", "https://yashash045-schemashift.hf.space")
health_client = SchemaShiftEnvClient(base_url=SCHEMASHIFT_URL)
if not health_client.health():
    raise RuntimeError(
        f"Env at {SCHEMASHIFT_URL} is not responding to /health. "
        "Fix SCHEMASHIFT_URL secret or redeploy the Space before training."
    )
print(f"[OK] Env reachable at {SCHEMASHIFT_URL}")
tasks = health_client.list_tasks()
print(f"[OK] {len(tasks)} scenarios available: {[t['task_id'] for t in tasks]}")
health_client.close()


In [ ]:
# Load Qwen 2.5 1.5B + LoRA adapter.
# fast_inference=False disables Unsloth's vLLM bypass - required when vLLM isn't installed.
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=3072,
    load_in_4bit=True,
    fast_inference=False,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
print(f"Trainable params: {model.num_parameters(only_trainable=True):,}")


In [ ]:
# Env client + reward function (variant-specific  -  uncomment ONE block)
import os, json, re
from client import SchemaShiftEnvClient
from models import Action, CompleteParams

SCHEMASHIFT_URL = os.getenv("SCHEMASHIFT_URL", "https://YOUR-SPACE.hf.space")
env_client = SchemaShiftEnvClient(base_url=SCHEMASHIFT_URL)


def parse_completion_to_actions(text: str) -> list[Action]:
    """Parse LLM output into a list of Actions. Graceful fallback on any failure."""
    # Strip common markdown code fences
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", text)
    actions = []
    # Brace-balanced extractor (handles arbitrary nesting, unlike flat regex)
    depth = 0
    start = -1
    for i, ch in enumerate(cleaned):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start >= 0:
                chunk = cleaned[start:i + 1]
                try:
                    obj = json.loads(chunk)
                    actions.append(Action.model_validate(obj))
                except Exception:
                    pass
                start = -1
    if not actions:
        actions.append(Action(
            type="complete_task",
            complete=CompleteParams(summary="Parse error  -  no valid actions in output."),
        ))
    return actions


# --- ACCOUNT 1 (Yashash - STAGE 1 MAIN): shaped_total (rubric + dense + gates) ---
def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        actions = parse_completion_to_actions(completion)
        task_id = kwargs.get("task_id", "E1_onboard_new_hire")
        env_client.reset(task_id)
        total = 0.0
        for a in actions:
            obs, r = env_client.step(a)
            total += r.shaped_total
            if obs.done:
                break
        rewards.append(total)
    return rewards


# --- ACCOUNT 2 (Gajanand - Stage 2 parallel, shaped_total - same as Account 1) ---
# def reward_fn(prompts, completions, **kwargs):
#     rewards = []
#     for prompt, completion in zip(prompts, completions):
#         actions = parse_completion_to_actions(completion)
#         task_id = kwargs.get("task_id", "E1_onboard_new_hire")
#         env_client.reset(task_id)
#         total = 0.0
#         for a in actions:
#             obs, r = env_client.step(a)
#             total += r.shaped_total
#             if obs.done:
#                 break
#         rewards.append(total)
#     return rewards


# --- ACCOUNT 3 (Likith - curriculum ablation): shaped + procedural scheduler ---
# See procedural scheduler cell below. Rebuild dataset every ~25 training steps.
# Alternative for Account 3: swap Cell 4 (model load) to 'Qwen/Qwen2.5-Coder-1.5B-Instruct'.


In [ ]:
# STAGE 1: skipped - uncomment for Account 3 Stage 2+ curriculum.
# (Defining this function has no runtime effect for Accounts 1 & 2; commenting
# to make the skip intent explicit.)

# Procedural drift scheduler (Account 3 only  -  comment out for Accounts 1 & 2)
# import random
# from copy import deepcopy
# from scenarios import SCENARIOS
# from models import DriftEvent


# def build_procedural_scenarios(step_count: int) -> dict:
#     """Curriculum-based drift variety that scales with training progress.

#     - Steps 0-50:   Original deterministic drifts (baseline)
#     - Steps 50-150: Permute drift steps (same drifts, shuffled timing)
#     - Steps 150+:   Add a secondary drift (harder, multi-drift episodes)
#     """
#     scenarios_out = {}
#     for task_id, scenario in SCENARIOS.items():
#         if scenario["difficulty"] != "easy":
#             scenarios_out[task_id] = scenario
#             continue

#         s = deepcopy(scenario)

#         if step_count < 50:
#             pass
#         elif step_count < 150:
#             max_step = s["max_steps"]
#             for d in s["drift_plan"]:
#                 d.fires_at_step = random.randint(1, max(1, max_step - 2))
#                 d.details["_fired"] = False
#         else:
#             if s["drift_plan"]:
#                 primary = s["drift_plan"][0]
#                 secondary_kinds = ["rate_limit_tightening", "error_code_remap"]
#                 secondary = DriftEvent(
#                     tool=primary.tool,
#                     endpoint=None,
#                     kind=random.choice(secondary_kinds),
#                     fires_at_step=min(s["max_steps"] - 1, primary.fires_at_step + 2),
#                     details={"_procedural": True},
#                 )
#                 s["drift_plan"].append(secondary)

#         scenarios_out[task_id] = s
#     return scenarios_out

In [ ]:
# Prompt dataset builder
from datasets import Dataset
from scenarios import SCENARIOS


def build_prompts_dataset(scenarios_dict):
    prompts = []
    for task_id, scenario in scenarios_dict.items():
        prompt = f"""You are an autonomous workflow agent. Complete the task using tools.

TASK: {scenario['task_description']}

SUCCESS CRITERIA:
{chr(10).join(f'- {c}' for c in scenario['success_criteria'])}

AVAILABLE ACTIONS:
- call_tool(tool, endpoint, params) - call a tool
- inspect_schema(tool) - read current schema for a tool
- retry_with_variant(tool, endpoint, params) - retry a failed call with modified params
- report_drift(tool, drift_kind, description) - flag a drift you detected
- complete_task(summary) - end the episode

Output your next action as JSON. Example:
{{"type": "call_tool", "tool_call": {{"tool": "mail", "endpoint": "send_message", "params": {{"to": "x@y.com", "subject": "Hi", "body": "Hello"}}}}}}
"""
        prompts.append({"prompt": prompt, "task_id": task_id})
    return prompts


# Stage 1 main: all 6 scenarios as a HuggingFace Dataset (GRPOTrainer expects this shape).
train_dataset = Dataset.from_list(build_prompts_dataset(SCENARIOS))
print(f"{len(train_dataset)} scenarios loaded")

# Account 3 (procedural - rebuild each training chunk):
# train_dataset = Dataset.from_list(build_prompts_dataset(build_procedural_scenarios(step_count=0)))


In [ ]:
# GRPO config (TRL 0.29.0, Kaggle T4-friendly, no vLLM).
# Adopts winner's DAPO loss + cosine LR + light KL - minus vLLM/rollout_func
# (rollout_func only runs when use_vllm=True in TRL 0.29.0; we keep env rollout
# inside reward_fn which TRL calls after its internal model.generate()).
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="schemashift-grpo-kaggle",
    num_generations=4,
    max_completion_length=1024,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    max_grad_norm=1.0,
    logging_steps=1,
    save_strategy="steps",
    save_steps=25,
    max_steps=100,
    temperature=1.0,
    report_to="none",
    use_vllm=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    loss_type="dapo",
    mask_truncated_completions=True,
    beta=0.01,
    save_total_limit=3,
    push_to_hub=True,
    hub_model_id=f"{os.getenv('HF_USERNAME', 'yashash045')}/schemashift-qwen15b-kaggle",
    hub_strategy="every_save",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=train_dataset,
)

print("GRPOTrainer initialized (TRL 0.29.0, use_vllm=False).")
print(f"  model:         Qwen/Qwen2.5-1.5B-Instruct")
print(f"  dataset size:  {len(train_dataset)}")
print(f"  max_steps:     {config.max_steps}")
print(f"  save_steps:    {config.save_steps}")
print(f"  loss_type:     {config.loss_type}")
print(f"  hub_model_id:  {config.hub_model_id}")


In [ ]:
# Train.
# Go/no-go watch window: see STAGE_1_KAGGLE_RUNBOOK.md step 5.
# If first 5 logged mean_rewards are all 0.0, STOP - parser or reward path broken.
trainer.train()


In [ ]:
# Save + push final checkpoint.
# (push_to_hub=True in GRPOConfig auto-pushes at save_steps intervals;
# this final explicit push captures state past the last save_step.)
trainer.save_model(config.output_dir)
trainer.push_to_hub(commit_message="Stage 1 final - Run 1 - Account 1")
print("Training complete. Final checkpoint saved locally and pushed to HF Hub.")
